In [2]:
%pip install stim

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 2.1 MB/s eta 0:00:02
   ----------- ---------------------------- 0.8/2.8 MB 2.0 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.8 MB 1.7 MB/s eta 0:00:01
   -------------------------- ------------- 1.8/2.8 MB 1.8 MB/s eta 0:00:01
   ---------------------------------- ----- 2.4/2.8 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 1.9 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import stim
import numpy as np
import os

def generate_large_scale_dataset(distance=5, rounds=5, num_shots=100000, physical_error_rate=0.03):
    print(f"Initializing Stim pipeline for Distance-{distance}, {rounds} Rounds...")
    print(f"Simulating {num_shots} shots at a {physical_error_rate * 100}% error rate...")

    # 1. Generate the standardized noisy QEC circuit
    # FIXED: Replaced 'measurement_error' with 'before_measure_flip_probability'
    circuit = stim.Circuit.generated(
        "repetition_code:memory", # Using 'memory' is standard for tracking logical state survival over time
        distance=distance,
        rounds=rounds,
        after_clifford_depolarization=physical_error_rate, 
        before_measure_flip_probability=physical_error_rate
    )

    # 2. Compile Stim's high-speed sampler
    sampler = circuit.compile_detector_sampler()

    # 3. Sample the system
    # X_data: The 20-bit sparse syndrome vectors (Features)
    # y_data: The binary tracking of the final logical state flip (Target)
    X_data, y_data = sampler.sample(shots=num_shots, separate_observables=True)

    # 4. Cast to standard ML formats (Float32 for Neural Network compatibility)
    X_data = X_data.astype(np.float32)
    y_data = y_data.astype(np.float32).flatten()

    # 5. Save the arrays locally to prevent re-generation overhead
    os.makedirs("data", exist_ok=True)
    np.save("data/X_stim_syndromes.npy", X_data)
    np.save("data/y_stim_labels.npy", y_data)

    print("\n--- STIM GENERATION SUCCESSFUL ---")
    print(f"Saved Features (X) Shape: {X_data.shape} -> ({num_shots} snapshots, {X_data.shape[1]} features each)")
    print(f"Saved Targets  (y) Shape: {y_data.shape} -> ({num_shots} binary ground-truth states)")
    print("Files successfully cached inside the 'data/' directory.")

if __name__ == "__main__":
    generate_large_scale_dataset(distance=5, rounds=5, num_shots=100000, physical_error_rate=0.03)

Initializing Stim pipeline for Distance-5, 5 Rounds...
Simulating 100000 shots at a 3.0% error rate...


TypeError: generated(): incompatible function arguments. The following argument types are supported:
    1. (code_task: str, *, distance: int, rounds: int, after_clifford_depolarization: float = 0.0, before_round_data_depolarization: float = 0.0, before_measure_flip_probability: float = 0.0, after_reset_flip_probability: float = 0.0) -> stim._stim_sse2.Circuit

Invoked with: 'repetition_code:bits'; kwargs: distance=5, rounds=5, after_clifford_depolarization=0.03, measurement_error=0.03